# Topic 6 - PEFT and LoRA with DistilBERT

Barclays Customer Support Intelligence System | Topic 6

## What you will build

This topic teaches you how LoRA (Low-Rank Adaptation) works and how to apply it with
the production-grade HuggingFace PEFT library. A short mini-lesson below explains the
mechanics - low-rank decomposition, and what rank, alpha, and dropout actually do - so
you can reason about the knobs you tune.

You then use the PEFT library to apply LoRA to a full DistilBERT complaint classifier,
the same pattern used in real ML pipelines at scale. If you want to see LoRA built by
hand on a feed-forward network, there is an optional deep-dive notebook
(topic_optional_lora_ffn); it is not required for this topic.
You then explore QLoRA (4-bit quantized base plus LoRA adapters) and soft prompts as
alternative parameter-efficient strategies, and close with an open-ended capstone
where you design and launch your own PEFT training job on a GPU instance.

## Why this matters to Barclays

Fine-tuning a full 66M-parameter DistilBERT on Barclays hardware budgets is expensive.
PEFT methods reduce trainable parameters by 99% or more while matching full fine-tune
accuracy. QLoRA further cuts GPU memory by 4x, making GPU training accessible on
smaller instances. These are the techniques Barclays ML teams use in production today.

## Learning objectives

1. Apply the PEFT library (LoraConfig plus get_peft_model) to any HuggingFace encoder model
2. Explain QLoRA: how 4-bit NF4 quantization combines with LoRA adapters
3. Describe soft prompts / prefix tuning and when to prefer them over LoRA
4. Launch a GPU fine-tuning job on SageMaker using the HuggingFace estimator
5. Compare trainable parameter counts across LoRA, QLoRA, and full fine-tuning

## Estimated time

90 to 120 minutes in class.

In [ ]:
# Disable TensorFlow backend in transformers (SageMaker image compatibility).
# Must run before any transformers import.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


In [ ]:
# Environment setup for SageMaker Studio.
# CPU kernel is used for all demos in Sections 1-3.
# The GPU training job in Section 4 runs remotely on ml.g4dn.xlarge.
# bitsandbytes and PEFT are installed in the remote container via requirements.txt.
# We do NOT install bitsandbytes in this notebook kernel (CPU only, would error).

!pip install -q "sagemaker>=2.200.0,<3.0.0" \
    "transformers>=4.53,<4.54" \
    "accelerate>=1.0.0" \
    "tokenizers>=0.21,<0.22" \
    "datasets>=2.18.0,<3.0.0" \
    "peft>=0.6.0" \
    "numpy<2"

print("RESTART KERNEL before continuing -- environment packages were installed/upgraded.")


In [ ]:
import sagemaker
from sagemaker import get_execution_role
import boto3

sess   = sagemaker.Session()
role   = get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name

print(f"Role:   {role}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")

In [ ]:
# Handoff from Topic 5: load the transfer-learned model pointer you produced there.
# In Topic 5 you trained a transfer-learning classifier and saved its S3 URI.
# We load it now and extend the system: this topic adapts the same classifier with
# parameter-efficient fine-tuning (PEFT and LoRA).

# --- S3 handoff helpers ---
import json, boto3, botocore

COURSE_PREFIX = "barclays-course"

def handoff_read(bucket, topic_n, artifact):
    key = f"{COURSE_PREFIX}/topic_{topic_n}/{artifact}"
    try:
        body = boto3.client("s3").get_object(Bucket=bucket, Key=key)["Body"].read()
        print(f"Handoff loaded: s3://{bucket}/{key}")
        return json.loads(body)
    except botocore.exceptions.ClientError:
        print(f"No handoff found at s3://{bucket}/{key} (starting mid-course is fine).")
        return None

_t5 = handoff_read(bucket, 5, "model_pointer.json")

if _t5 is not None:
    prior_model_uri = _t5.get("model_tar_uri")
    label_map = {int(k): v for k, v in _t5["label_map"].items()}
    print(f"Loaded Topic 5 transfer-learning pointer; {len(label_map)} labels.")
else:
    # Fallback: student is starting at Topic 6. Use the 4-class Barclays routing
    # label map so this topic stays consistent with the rest of the course.
    print("Using a local fallback label map so Topic 6 runs standalone.")
    prior_model_uri = None
    label_map = {
        0: "fraud and security",
        1: "billing and charges",
        2: "account access",
        3: "general enquiry",
    }

num_labels = len(label_map)
print(f"PEFT/LoRA target: {num_labels}-class Barclays complaint classifier.")

## Where This Topic Fits

We are building the Barclays Customer Support Intelligence System end to end.
Each topic adds one layer. Today you are here:

| Step | Topic | What it adds to the system |
|------|-------|---------------------------|
| 1 | Topic 3 HuggingFace | Load pre-trained models from the Hub |
| 2 | Topic 4 Full Fine-Tuning | Adapt a model to Barclays complaints |
| 3 | Topic 5 Transfer Learning | Freeze the encoder, train only the head |
| 4 | Topic 6 PEFT and LoRA | Apply the PEFT library to a full classifier (YOU ARE HERE) |
| 5 | Topic 7 Quantization | Compress and deploy the model |

By the end of the required path you will have a fine-tuned, PEFT-adapted DistilBERT
complaint classifier compressed and running as a SageMaker endpoint.

In [ ]:
import torch
import numpy as np
import warnings
import os
import random

warnings.filterwarnings("ignore")

def set_seeds(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

# CPU kernel for demos. GPU job runs remotely.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")

# LoRA decomposes a weight update into two small matrices A and B of rank r.
# The mini-lesson below explains rank, alpha, and dropout in detail.
lora_r = 8   # rank: a small r means very few trainable parameters
print(f"LoRA rank: {lora_r}")

## LoRA Mechanics -- A Concept Mini-Lesson

Before you call the PEFT library, you need to understand what it is doing, so you can
reason about the knobs you tune. This mini-lesson is concept-level. If you want to see
LoRA implemented by hand on a feed-forward network, the optional deep-dive notebook
topic_optional_lora_ffn does exactly that; it is not required here.

### The problem: full fine-tuning updates everything

Fine-tuning a transformer the full way (Topic 4) updates every weight matrix W. For
DistilBERT that is about 66 million parameters, and the Adam optimizer needs extra
memory for each one. That is expensive, and you get a whole new copy of the model per
task.

### The idea: the UPDATE is low-rank

Fine-tuning changes a weight matrix W into W + deltaW. The key empirical finding behind
LoRA (Hu et al., 2021) is that deltaW - the CHANGE - does not need to be full rank. It
can be well approximated by the product of two much smaller matrices:

  deltaW (shape d by k)  is approximated by  B (shape d by r) times A (shape r by k)

Here r, the RANK, is small - typically 4, 8, or 16. Instead of training all d times k
numbers in deltaW, you train only r times (d + k) numbers in A and B. For a 768 by 768
attention projection with r = 8, that is about 12,000 trainable numbers instead of
about 590,000 - roughly 50 times fewer.

During training W is FROZEN. Only A and B are learned. At inference the layer computes:

  output = W x  +  (B A) x

A is initialized random, B is initialized to zero, so at the start B A = 0 and the
model behaves exactly like the pretrained one - training then nudges it from there.

In [ ]:
# Concept demo: how many parameters does LoRA actually train?
# This counts parameters for one attention projection. It is an illustration of the
# mechanics, not the real injection (the PEFT library does that later in this notebook).

d, k = 768, 768   # a DistilBERT attention projection is 768 x 768

full_finetune = d * k                      # every number in delta-W

for r in [4, 8, 16, 32]:
    # LoRA trains A (r x k) and B (d x r) instead of the full delta-W.
    lora_params = r * k + d * r
    ratio = full_finetune / lora_params
    print(f"rank r = {r:>2}:  LoRA trains {lora_params:>8,} params  "
          f"vs {full_finetune:,} full  ->  {ratio:5.1f}x fewer")

print()
print("Bigger r = more capacity to fit the new task, but more parameters and more")
print("risk of overfitting a small dataset. Smaller r = cheaper and more regularized.")
print("r = 8 is the common default for DistilBERT-sized models.")

### The Three Knobs You Tune

**rank (r)**: the inner dimension of A and B. It sets how much the adapter can change
the model. Small r (4-8) is cheap and resists overfitting on small datasets; larger r
(16-32) gives more capacity for harder or more distant tasks. The demo above shows the
parameter cost of each choice.

**lora_alpha**: a scaling factor. The adapter's contribution is scaled by alpha / r
before it is added to the frozen path. Think of it as a learning-rate-like gain on the
adapter: raising alpha makes the adapter's effect stronger, lowering it makes the
adapter more conservative. A common convention is to set alpha = 2 times r, so the
effective scale stays steady as you change r.

**lora_dropout**: ordinary dropout applied to the adapter input during training. It
randomly zeroes some activations so the adapter cannot rely on any single feature,
which regularizes a small adapter trained on a small dataset. Typical values are 0.05
to 0.1. It is active during training only and does nothing at inference.

### Why it works

The pretrained model already knows language. Adapting it to Barclays complaints is a
small, low-dimensional shift, not a from-scratch relearning - so a low-rank deltaW is
enough. Because W stays frozen, the original knowledge cannot be overwritten: this is
why LoRA does not suffer the catastrophic forgetting you saw in Topic 4. And because
A and B are tiny, you can store one small adapter per task instead of a full model copy.

**Peer discussion (3-5 min)**: You fine-tune a DistilBERT complaint classifier with
LoRA at r = 4 and accuracy plateaus below full fine-tuning. Would you raise r, raise
lora_alpha, or both - and what is the risk of each? When would you instead leave r low
on purpose?

## CUDA Health Check

This kernel may be running on a GPU instance or a CPU one depending on quota.
The cell below checks whether a GPU is present and whether `torch` can use it.
If a GPU is present but `torch` is a CPU-only build, it reinstalls a matching
CUDA wheel. If you see a reinstall message, restart the kernel before continuing.


In [ ]:
# CUDA health check + safety-net.
# A SageMaker kernel may land on a GPU instance (ml.g4dn.xlarge) or a CPU one
# (ml.t3.medium) depending on quota. On a GPU box the image's torch is sometimes
# CPU-only, so torch.cuda.is_available() is wrongly False. This cell detects the
# real situation from two signals (nvidia-smi + the SageMaker metadata file) and
# reinstalls a matching CUDA wheel only when needed.
import json
import os
import re
import shutil
import subprocess
import sys

import torch


def _sagemaker_image():
    # SageMaker writes the running image/instance here. Diagnostic only.
    path = "/opt/ml/metadata/resource-metadata.json"
    if not os.path.exists(path):
        return None
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def _nvidia_smi():
    # Returns (gpu_present, driver_cuda_version_str_or_None).
    # nvidia-smi exit 0 means GPU hardware is here even if torch cannot see it.
    if shutil.which("nvidia-smi") is None:
        return False, None
    try:
        out = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=15)
    except Exception:
        return False, None
    if out.returncode != 0:
        return False, None
    m = re.search(r"CUDA Version:\s*([0-9]+\.[0-9]+)", out.stdout)
    return True, (m.group(1) if m else None)


def _pick_wheel(driver_cuda):
    # Newest cuXXX wheel index <= the driver's max CUDA.
    # SageMaker g4dn (T4) images historically ship CUDA 12.1-12.4 drivers.
    tiers = [("12.6", "cu126"), ("12.4", "cu124"), ("12.1", "cu121")]
    if not driver_cuda:
        return "cu121"  # safe default when the driver version is unreadable
    try:
        dv = tuple(int(x) for x in driver_cuda.split("."))
    except ValueError:
        return "cu121"
    for ver, tag in tiers:
        if tuple(int(x) for x in ver.split(".")) <= dv:
            return tag
    return "cu121"


meta = _sagemaker_image()
if meta:
    print("SageMaker kernel:", meta.get("AppType", "?"),
          "| instance:", meta.get("ResourceArn", "?").split("/")[-1])

has_gpu, driver_cuda = _nvidia_smi()

if not has_gpu:
    print("No GPU detected. Running on CPU kernel. No action needed.")
elif torch.version.cuda is not None:
    print(f"GPU OK. torch {torch.__version__} built for CUDA {torch.version.cuda}.")
    print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
else:
    wheel = _pick_wheel(driver_cuda)
    print(f"GPU present (driver CUDA {driver_cuda}) but torch is CPU-only.")
    print(f"Reinstalling torch from the {wheel} wheel index...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade",
         "torch", "torchvision", "torchaudio",
         "--index-url", f"https://download.pytorch.org/whl/{wheel}"],
        check=True,
    )
    print("Reinstalled torch with CUDA support.")
    print("RESTART THE KERNEL now, then re-run the notebook from the top.")


## Beat 1 -- Section 1: From Scratch to Library, PEFT LoRA on DistilBERT


From the mini-lesson above, LoRA looks like this:

  output = W_frozen @ x + (B @ A) @ x

W_frozen stays fixed; only the small matrices A and B are trained. Doing that injection
by hand for every attention projection is tedious and error-prone. The PEFT library
automates it for any HuggingFace model. Three function calls replace two custom classes
and a manual layer-replacement loop. (The optional notebook topic_optional_lora_ffn
shows the by-hand build in full.)

### Beat 1: What happens if we try to apply LoRA without PEFT?

In [ ]:
# Beat 1: Trying to inject LoRA manually into DistilBERT
# (the approach from 7a) breaks on the classification head.
# DistilBERT's classifier is a separate module, manual replacement
# of q_lin / v_lin does NOT freeze it, causing all parameters to train.

from transformers import AutoModelForSequenceClassification
import torch.nn as nn

model_name = "distilbert-base-uncased"
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=4
)

# Manual attempt: replace q_lin in each attention block with a custom linear.
# This is the hand-rolled style. Let's see why it is incomplete here.

class NaiveLoraLinear(nn.Module):
    """Hand-rolled LoRA wrapper (the optional-deep-dive style), applied naively to DistilBERT."""
    def __init__(self, original_linear, rank=8):
        super().__init__()
        in_f  = original_linear.in_features
        out_f = original_linear.out_features
        self.frozen_W = original_linear
        self.frozen_W.weight.requires_grad_(False)
        self.A = nn.Linear(in_f,  rank,  bias=False)
        self.B = nn.Linear(rank,  out_f, bias=False)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.frozen_W(x) + self.B(self.A(x))

# Inject into the first attention block only (naive, partial).
first_attn = base_model.distilbert.transformer.layer[0].attention
first_attn.q_lin = NaiveLoraLinear(first_attn.q_lin)

# Count trainable parameters.
trainable = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in base_model.parameters())
print(f"Naive manual injection: {trainable:,} / {total:,} trainable")
print("Problem: the classifier head is still fully trainable.")
print("Problem: we only touched ONE attention block, missed all others.")
print("Problem: no easy way to save/load just the LoRA weights.")
print()
print("The PEFT library solves all three problems in 5 lines of code.")

## Beat 2 -- Diagram
<!-- DIAGRAM: PEFT methods comparison (LoRA, QLoRA, Soft Prompts) showing parameter efficiency of each -->
[View diagram](../../plans/topic_6_peft_lora_distilbert/diagrams/peft-methods-comparison.mmd)

The diagram above shows how each PEFT method modifies (or does not modify) the weight
matrices of a transformer block, and the relative trainable parameter count for each.
Notice that all three methods keep the base model frozen, only the coloured elements
(adapters or virtual tokens) are learned.

In [ ]:
# Beat 3: PEFT library LoRA on DistilBERT for sequence classification.
# This replaces the entire hand-rolled manual injection with 5 lines.

from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

model_name = "distilbert-base-uncased"

# Step 1: Load the base model. Frozen by PEFT automatically.
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=4   # 4 Barclays routing classes (see handoff cell)
)

# Step 2: Define the LoRA configuration.
# target_modules: DistilBERT attention projections are named q_lin and v_lin.
# modules_to_save: classifier head must also be trained (it is task-specific).
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,    # sequence classification task
    r=lora_r,                      # rank r=8 (see the LoRA mini-lesson above)
    lora_alpha=16,                 # scaling factor (alpha/r = effective lr scale)
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["classifier", "pre_classifier"],  # keep these trainable
)

# Step 3: Wrap the model. This freezes base weights and injects LoRA adapters.
peft_model = get_peft_model(base_model, lora_config)

# Step 4: Inspect, compare parameter counts.
peft_model.print_trainable_parameters()

trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in peft_model.parameters())
print(f"\nTrainable : {trainable:,}")
print(f"Total     : {total:,}")
print(f"Ratio     : {100.0 * trainable / total:.2f}%")
print()
print("Notice: LoRA reduces trainable params dramatically.")
print("The adapter weights for q_lin and v_lin across all 6 blocks")
print("plus the full classifier head make up the trainable fraction.")

In [ ]:
# Verify peft_model runs correctly on a dummy batch.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample_texts = [
    "Someone used my card to make payments I did not authorise.",
    "You charged me an overdraft fee I did not expect.",
    "I cannot log in to the mobile banking app.",
    "What are your branch opening hours this weekend?",
]

inputs = tokenizer(
    sample_texts,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=64,
)

peft_model.eval()
with torch.no_grad():
    outputs = peft_model(**inputs)

logits = outputs.logits
predictions = torch.argmax(logits, dim=-1)
demo_label_map = {0: "fraud and security", 1: "billing and charges",
                  2: "account access", 3: "general enquiry"}

print("PEFT LoRA DistilBERT, inference check")
print("-" * 42)
for text, pred in zip(sample_texts, predictions.tolist()):
    print(f"  [{demo_label_map[pred]}]  {text[:55]}")
print()
print("Forward pass succeeded. Adapters injected correctly.")

## Beat 4 -- Lab 1: Apply PEFT LoRA with a Different Rank (Tier 1 Guided, ~15 min)


### Situation
The Barclays complaint classification team asks: does a higher LoRA rank always give
us more trainable parameters? How does alpha affect the effective learning rate?

### Task
Create a second LoRA-wrapped DistilBERT with rank r=16 and alpha=32 (double both values).
Count trainable parameters and compare to the r=8 model above.

### Action
Fill in the YOUR CODE sections below.

### Result
A verification cell will print both parameter counts and the ratio.

### Steps
1. Load a fresh AutoModelForSequenceClassification (distilbert-base-uncased, num_labels=4).
2. Build a LoraConfig with r=16, lora_alpha=32, same target_modules and modules_to_save.
3. Wrap with get_peft_model().
4. The verification cell counts trainable parameters for you.

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

# Lab 1: Build a LoRA model with r=16 and compare parameter counts to peft_model (r=8).

# Step 1: load a fresh base model.
base_model_r16 = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=4
)

# Step 2: define LoraConfig with r=16, lora_alpha=32.
lora_config_r16 = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["classifier", "pre_classifier"],
)

# Step 3: wrap with get_peft_model.
peft_model_r16 = get_peft_model(base_model_r16, lora_config_r16)
peft_model_r16.print_trainable_parameters()

In [ ]:
# Lab 1 safety-net: run this if you did not finish Lab 1.
# SKIP this cell if you DID finish Lab 1.
if peft_model_r16 is None:
    print("Using Lab 1 safety-net so the rest of the notebook can run.")
    _base = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=4
    )
    _cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=16,
        lora_alpha=32,
        target_modules=["q_lin", "v_lin"],
        lora_dropout=0.1,
        bias="none",
        modules_to_save=["classifier", "pre_classifier"],
    )
    peft_model_r16 = get_peft_model(_base, _cfg)

In [ ]:
# Lab 1 verification, auto-graded.
t_r8  = sum(p.numel() for p in peft_model.parameters()     if p.requires_grad)
t_r16 = sum(p.numel() for p in peft_model_r16.parameters() if p.requires_grad)

print(f"LoRA r=8  trainable params : {t_r8:,}")
print(f"LoRA r=16 trainable params : {t_r16:,}")
print(f"r=16 / r=8 ratio           : {t_r16 / t_r8:.2f}x")

assert t_r16 > t_r8, "r=16 should have more trainable params than r=8"
assert 1.05 < (t_r16 / t_r8) < 2.5, "ratio should reflect doubling of LoRA rank"
print("Verification passed.")

### Stretch (for fast finishers)

Try r=4 with lora_alpha=8. Does the accuracy gap between r=4 and r=16 matter more than
the parameter count savings? Write your hypothesis in a markdown cell, then test it in
the Capstone (Section 4) by submitting two SageMaker jobs with different ranks.

### Homework Extension

Research the relationship between lora_alpha / r (the effective scaling ratio) and the
learning rate. Plot trainable parameter count vs rank (r = 2, 4, 8, 16, 32) for the
DistilBERT classification head. At what rank do diminishing returns set in?

In [ ]:
# Homework Extension starter: trainable params vs rank plot.
# Sweep r in [2, 4, 8, 16, 32] and record peft_model.print_trainable_parameters().
import matplotlib.pyplot as plt

ranks = [2, 4, 8, 16, 32]
trainable_counts = []
for r in ranks:
    base = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=4
    )
    cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=r, lora_alpha=2 * r,
        target_modules=["q_lin", "v_lin"],
        lora_dropout=0.1, bias="none",
        modules_to_save=["classifier", "pre_classifier"],
    )
    m = get_peft_model(base, cfg)
    trainable_counts.append(
        sum(p.numel() for p in m.parameters() if p.requires_grad)
    )

plt.figure(figsize=(6, 4))
plt.plot(ranks, trainable_counts, marker="o")
plt.xlabel("LoRA rank r")
plt.ylabel("Trainable parameters")
plt.title("DistilBERT PEFT LoRA: trainable params vs rank")
plt.grid(True)
plt.show()

## Discussion (3 minutes)

You just reduced DistilBERT's trainable parameters from 66M to roughly 300K using LoRA.

1. If a Barclays team is training on 10K labelled complaints, does the rank r=8 vs r=16
   difference matter? What signals would you look for to decide?

2. LoRA adapters are stored separately from the base model weights. What does that mean
   for model versioning, rollback, and A/B testing at Barclays?

3. The base weights are frozen during LoRA training. What happens if the base model is
   updated (e.g., a new version of DistilBERT is released)? Do you retrain the adapters?

## Beat 3 -- Section 2: QLoRA, 4-Bit Quantization plus LoRA


QLoRA (Dettmers et al., 2023) combines two ideas:
  1. Quantize the base model weights to 4-bit NF4 (NormalFloat4), cuts memory ~4x.
  2. Apply LoRA adapters in float16 on top of the frozen 4-bit base.

Gradients flow only through the LoRA adapters. The 4-bit base weights are never updated.
For DistilBERT (66M params), this means the base takes ~22MB instead of ~268MB (float32).
For larger models (7B params), QLoRA makes GPU training feasible on a single T4.

### Beat 1: What happens when you try QLoRA without a GPU?

In [ ]:
# Beat 1: bitsandbytes requires a CUDA GPU. On this CPU kernel it will fail.
# This is intentional, students see the exact error before the solution.

import sys

try:
    import bitsandbytes as bnb
    print(f"bitsandbytes version: {bnb.__version__}")
    # Even if the import works, 4-bit quantization requires CUDA at model load time.
    from transformers import BitsAndBytesConfig, AutoModelForSequenceClassification
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
    )
    # This will raise RuntimeError or ValueError on CPU:
    #   "bitsandbytes is only supported on CUDA devices"
    fail_model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=4, quantization_config=bnb_config
    )
except (ImportError, RuntimeError, ValueError) as e:
    print(f"[Expected error on CPU kernel] {type(e).__name__}: {e}")
    print()
    print("bitsandbytes requires a CUDA GPU, it cannot run on this CPU kernel.")
    print("Solution: move QLoRA training to a GPU instance via SageMaker.")
    print("The scripts_topic6/train.py script uses QLoRA when --peft_method=qlora.")

<!-- DIAGRAM: QLoRA architecture showing 4-bit NF4 base model with float16 LoRA adapters -->
[View diagram](../../plans/topic_6_peft_lora_distilbert/diagrams/qlora-architecture.mmd)

The diagram above shows the QLoRA forward pass. The base model weights (grey) are stored
in 4-bit NF4 format. Before each matrix multiply, they are dequantized to float16 on
the fly. LoRA adapters (A and B) stay in float16 throughout. Gradients only flow through
the adapters, the 4-bit base is never updated.

In [ ]:
# Beat 3: Full QLoRA code, explained line by line.
# This cell does NOT run on the CPU kernel (no bitsandbytes without CUDA).
# Students read and understand it here. It runs inside scripts_topic6/train.py
# on the GPU instance.

# Instructor narration: walk through this cell line by line.

# from transformers import BitsAndBytesConfig, AutoModelForSequenceClassification
# from peft import LoraConfig, get_peft_model, TaskType

# Step 1: Configure 4-bit quantization.
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,          # activate 4-bit loading
#     bnb_4bit_quant_type="nf4",  # NormalFloat4, optimal for normally distributed weights
#     bnb_4bit_compute_dtype=torch.float16,  # compute in fp16 (not 4-bit) during forward
#     bnb_4bit_use_double_quant=True,        # quantize the quantization constants too
# )

# Step 2: Load the base model in 4-bit.
# base_model = AutoModelForSequenceClassification.from_pretrained(
#     "distilbert-base-uncased",
#     num_labels=4,
#     quantization_config=bnb_config,  # triggers bitsandbytes 4-bit loading
#     device_map="auto",               # let bitsandbytes place layers optimally
# )

# Step 3: Apply LoRA adapters (identical config to the plain LoRA case above).
# lora_config = LoraConfig(
#     task_type=TaskType.SEQ_CLS,
#     r=8, lora_alpha=16,
#     target_modules=["q_lin", "v_lin"],
#     lora_dropout=0.1, bias="none",
#     modules_to_save=["classifier", "pre_classifier"],
# )
# qlora_model = get_peft_model(base_model, lora_config)
# qlora_model.print_trainable_parameters()
# # Output: trainable params: ~300K, all params: ~66M, trainable: ~0.46%
# # Base weights occupy only ~22MB in NF4 vs ~268MB in float32.

print("QLoRA code walkthrough complete.")
print("The full implementation runs in scripts_topic6/train.py --peft_method=qlora")
print("on the SageMaker GPU instance (ml.g4dn.xlarge, NVIDIA T4).")
print()
print("Key difference from plain LoRA:")
print("  LoRA   : base = float32 (~268MB), adapters = float32")
print("  QLoRA  : base = NF4 4-bit (~22MB), adapters = float16")
print("  Memory saved: ~10x reduction in base model footprint")

## Lab 2: Apply QLoRA to a Different DistilBERT Layer Configuration (Tier 2 Hard, 25-35 min)

This is a Tier 2 hard lab. You get a brief task description and YOUR CODE stubs in the
starter cell. Document your layer-choice reasoning in a markdown cell before you write
any code.

### Situation

The Barclays Model Efficiency team wants to evaluate whether targeting additional
attention layers (beyond q_lin and v_lin) in QLoRA gives a meaningful accuracy lift,
or whether the extra trainable parameters are wasted on a small dataset.

### Task

Implement the function `build_qlora_model` in the starter cell. Choose which DistilBERT
layers to target with LoRA, choose a rank, and explain why. Because QLoRA requires a GPU
(bitsandbytes), your implementation should gracefully fall back to plain LoRA on CPU so
the cell can run in this kernel. The docstring and comments describe the full QLoRA path
that would run on a GPU instance.

### What to hand in

- Your completed function in the code cell below
- A markdown cell (write it above the code cell) explaining:
    - Which target_modules you chose and why
    - What rank you chose and why
    - What tradeoffs you are making vs the default q_lin plus v_lin, r=8 config

### Constraints

- Fill in the YOUR CODE stubs in the starter cell
- No evaluate library, use inline numpy metrics if you compute accuracy
- numpy<2 requirement applies

In [ ]:
def build_qlora_model(
    model_name: str,
    target_modules: list,
    lora_r: int,
    lora_alpha: int,
    num_labels: int = 4,
):
    """
    Build a QLoRA-wrapped DistilBERT (or compatible encoder) for sequence classification.

    On a CUDA GPU, this loads the base model in 4-bit NF4 and applies LoRA adapters
    in float16. On CPU (this kernel), it falls back to plain float32 LoRA so the cell
    can execute without bitsandbytes.

    Returns (peft_model, trainable_params, total_params).
    """
    if torch.cuda.is_available():
        from transformers import BitsAndBytesConfig
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        base_model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            quantization_config=bnb_config,
            device_map="auto",
        )
    else:
        base_model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=num_labels,
        )

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=target_modules,
        lora_dropout=0.1,
        bias="none",
        modules_to_save=["classifier", "pre_classifier"],
    )
    peft_model = get_peft_model(base_model, lora_config)

    trainable_params = sum(
        p.numel() for p in peft_model.parameters() if p.requires_grad
    )
    total_params = sum(p.numel() for p in peft_model.parameters())

    return peft_model, trainable_params, total_params

In [ ]:
# Lab 2 safety-net: run this if you did not finish Lab 2.
# SKIP this cell if you DID finish Lab 2.
_probe = None
try:
    _probe = build_qlora_model("distilbert-base-uncased", ["q_lin", "v_lin"], 8, 16)
except Exception:
    _probe = None

if _probe is None:
    print("Using Lab 2 safety-net so the rest of the notebook can run.")
    def build_qlora_model(model_name, target_modules, lora_r, lora_alpha, num_labels=4):
        from transformers import AutoModelForSequenceClassification
        if torch.cuda.is_available():
            from transformers import BitsAndBytesConfig
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
            base_model = AutoModelForSequenceClassification.from_pretrained(
                model_name, num_labels=num_labels,
                quantization_config=bnb_config, device_map="auto",
            )
        else:
            base_model = AutoModelForSequenceClassification.from_pretrained(
                model_name, num_labels=num_labels,
            )
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=lora_r, lora_alpha=lora_alpha,
            target_modules=target_modules,
            lora_dropout=0.1, bias="none",
            modules_to_save=["classifier", "pre_classifier"],
        )
        peft_model = get_peft_model(base_model, lora_config)
        trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in peft_model.parameters())
        return peft_model, trainable, total
    _qm, _t, _tot = build_qlora_model("distilbert-base-uncased", ["q_lin", "v_lin"], 8, 16)
    print(f"Safety-net build_qlora_model -> trainable={_t:,}, total={_tot:,}")

### Stretch (for fast finishers)

Call build_qlora_model twice: once with your chosen layer set and once with only
q_lin plus v_lin (the default from the demo). Print both trainable parameter counts
side by side. Write a one-paragraph hypothesis about which configuration will
achieve higher validation accuracy on the 4-class Barclays routing task and why.

### Homework Extension

Submit two SageMaker jobs using scripts_topic6/train.py: one with your chosen
target_modules (pass them as a hyperparameter or edit train.py) and one with the
default q_lin plus v_lin. Compare eval_accuracy from CloudWatch. Write a short report
(3-5 sentences) explaining whether your hypothesis was correct and what you would
change in a second iteration.

In [ ]:
# Homework Extension starter: side-by-side parameter counts for two layer configs.
# Run after you finish Lab 2 (uses build_qlora_model).
config_a = ["q_lin", "v_lin"]                          # default
config_b = ["q_lin", "v_lin", "k_lin", "out_lin"]      # broader

_, t_a, total_a = build_qlora_model("distilbert-base-uncased", config_a, 8, 16)
_, t_b, total_b = build_qlora_model("distilbert-base-uncased", config_b, 8, 16)

print(f"{'Config':<40} {'Trainable':>12} {'% of total':>12}")
print("-" * 66)
print(f"{str(config_a):<40} {t_a:>12,} {100.0 * t_a / total_a:>11.3f}%")
print(f"{str(config_b):<40} {t_b:>12,} {100.0 * t_b / total_b:>11.3f}%")

## Peer Discussion (3 min)

Before we look at soft prompts, reflect on what we just saw with QLoRA:

1. QLoRA uses 4-bit quantization to cut memory by 75%. When would you NOT use it at Barclays, despite the savings?
2. LoRA adapts weight matrices; soft prompts adapt the input space. Which approach would you prefer for a model you cannot retrain from scratch?
3. If you had to deploy 20 different complaint-routing adapters for 20 Barclays departments, which technique stores more efficiently: LoRA, QLoRA, or soft prompts?

## Section 3: Soft Prompts, Learning Virtual Tokens

Soft prompts (also called prompt tuning or prefix tuning) take a completely different
approach from LoRA and QLoRA:

- Instead of modifying weight matrices, prepend P learnable virtual token embeddings
  to the input sequence at every transformer layer.
- The base model is 100% frozen, not even the attention weights change.
- Only the virtual token embeddings are trained (as few as 20 vectors).
- At inference time, prepend the same virtual tokens, no architecture change needed.

Tradeoff: Soft prompts use far fewer parameters than LoRA (~0.01-0.1% vs ~0.5-1%).
They work well for large models (7B+) but can underfit small models like DistilBERT.
LoRA typically outperforms soft prompts on small encoders for classification tasks.

### Beat 1: Shape mismatch when prefix length is wrong

In [ ]:
# Beat 1: A common mistake is setting num_virtual_tokens larger than max_length,
# which causes a shape mismatch inside the model's attention mask.

from peft import PromptTuningConfig, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification

# Load a fresh base model for soft prompt demo.
base_for_prompts = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=4
)

# Bad config: num_virtual_tokens=200 but our inputs use max_length=64.
# This will produce a RuntimeError inside the model forward pass.
bad_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,
    num_virtual_tokens=200,    # too many, 200 > max_length=64
    tokenizer_name_or_path="distilbert-base-uncased",
)

try:
    bad_model = get_peft_model(base_for_prompts, bad_config)
    # Trigger forward pass to expose the shape error.
    dummy_input = tokenizer(["test complaint"], return_tensors="pt", max_length=64,
                            truncation=True, padding="max_length")
    _ = bad_model(**dummy_input)
except Exception as e:
    print(f"[Expected error] {type(e).__name__}: {e}")
    print()
    print("num_virtual_tokens must be much smaller than max_sequence_length.")
    print("For max_length=64, a safe value is num_virtual_tokens=10 to 20.")

<!-- DIAGRAM: soft prompt / prefix tuning - P small trainable virtual token embeddings prepended to the input token sequence, feeding into a fully frozen DistilBERT; gradients flow only into the prompt embeddings, never the base model -->
[View diagram](../../plans/topic_6_peft_lora_distilbert/diagrams/soft_prompt_virtual_tokens.mmd)

The diagram shows how prompt tuning prepends a handful of learnable virtual token embeddings to the input sequence while the entire DistilBERT model stays frozen. Only those small prompt embeddings receive gradient updates during training.

In [ ]:
# Beat 3: Correct soft prompt / prefix tuning configuration.

from peft import PromptTuningConfig, PromptTuningInit, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification

base_for_prompts = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=4
)

# Correct config: num_virtual_tokens well below max_length.
prompt_config_demo = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,
    # Number of learnable virtual tokens prepended to the input.
    num_virtual_tokens=10,
    # Initialize virtual tokens from real vocabulary embeddings (faster convergence).
    prompt_tuning_init=PromptTuningInit.TEXT,
    prompt_tuning_init_text="route complaint to fraud billing access or enquiry",
    tokenizer_name_or_path="distilbert-base-uncased",
)

prefix_model = get_peft_model(base_for_prompts, prompt_config_demo)
prefix_model.print_trainable_parameters()

trainable = sum(p.numel() for p in prefix_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in prefix_model.parameters())
print(f"\nSoft prompt trainable : {trainable:,}")
print(f"Total                 : {total:,}")
print(f"Ratio                 : {100.0 * trainable / total:.4f}%")
print()
print("Notice: even fewer trainable params than LoRA.")
print("The 10 virtual tokens each have dim=768, 7,680 learned values in total.")

## Lab 1b: Configure a Soft Prompt for Barclays Complaint Routing (Tier 1 Guided, ~10 min)

### Situation
The Barclays team wants to test prefix tuning as a lightweight alternative to LoRA
for routing complaints to the right department (cards, payments, fraud).

### Task
Configure a PromptTuningConfig for the complaint classifier and verify the virtual
token count.

**Action:** Complete the YOUR CODE stubs below.
**Result:** Your soft prompt model should achieve comparable accuracy to LoRA with far fewer parameters.

### Steps
1. Create a PromptTuningConfig with task_type=SEQ_CLS, num_virtual_tokens=20.
2. Wrap a fresh base model with get_peft_model(model, config).
3. Count trainable parameters and confirm they are only the virtual tokens.

In [ ]:
from peft import PromptTuningConfig, TaskType, get_peft_model
from transformers import AutoModelForSequenceClassification

lab_base_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=4
)

# Step 1: create the config.
prompt_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,
    num_virtual_tokens=20,
)

# Step 2: wrap the base model.
prompt_model = get_peft_model(lab_base_model, prompt_config)

# Step 3: count trainable params.
trainable_soft = sum(
    p.numel() for p in prompt_model.parameters() if p.requires_grad
)
print(f"Trainable parameters: {trainable_soft:,}")

In [ ]:
# Lab 1b safety-net: run this if you did not finish Lab 1b.
# SKIP this cell if you DID finish Lab 1b.
if prompt_model is None:
    print("Using Lab 1b safety-net so the rest of the notebook can run.")
    prompt_config = PromptTuningConfig(
        task_type=TaskType.SEQ_CLS, num_virtual_tokens=20
    )
    prompt_model = get_peft_model(lab_base_model, prompt_config)
    trainable_soft = sum(
        p.numel() for p in prompt_model.parameters() if p.requires_grad
    )
    print(f"Safety-net trainable parameters: {trainable_soft:,}")

### Lab 1b Stretch: Tune Soft Prompt Hyperparameters

If you finish early, experiment with:
- Change `num_virtual_tokens` from 8 to 20 or 50 - how does accuracy change?
- Try `prompt_tuning_init=PromptTuningInit.TEXT` with a seed text like "Classify this customer complaint:"
- Compare soft prompt token count vs LoRA parameter count for the same model

### Homework Extension: Soft Prompt Transferability

**Barclays context:** Your team trained a soft prompt on complaint classification. Now a different product team wants to reuse it for a related task (routing tickets to departments).

**Task:** Investigate whether a soft prompt trained on one task transfers to a related task.

In [ ]:
# Homework: Test soft prompt transferability
# Step 1: Reuse the soft prompt model from Lab 1b (4-class Barclays routing)
# Step 2: Evaluate it zero-shot on a held-out set of paraphrased complaints
# Step 3: Compare against a random baseline over the 4 routing classes

import torch

# A held-out evaluation set: paraphrased Barclays complaints the soft prompt
# never saw during Lab 1b. Labels follow the 4-class routing scheme:
# 0 = fraud and security, 1 = billing and charges,
# 2 = account access, 3 = general enquiry.
holdout_texts = [
    "I spotted a payment on my statement that I never made.",
    "My card was charged for something I did not buy.",
    "An overdraft fee appeared that nobody warned me about.",
    "Why is there an extra service charge on my account?",
    "The banking app will not let me sign in at all.",
    "I am locked out after entering the wrong passcode.",
    "Can you tell me when my local branch opens?",
    "How do I order a replacement chequebook?",
]
holdout_labels = [0, 0, 1, 1, 2, 2, 3, 3]

# Evaluate the transferred soft prompt model zero-shot.
prompt_model.eval()
enc = tokenizer(
    holdout_texts, return_tensors="pt", truncation=True,
    padding=True, max_length=64,
)
with torch.no_grad():
    logits = prompt_model(**enc).logits
preds = logits.argmax(dim=-1).tolist()
transferred_accuracy = sum(p == l for p, l in zip(preds, holdout_labels)) / len(holdout_labels)

# For comparison: a random baseline over the 4 routing classes.
import random
random.seed(42)
random_preds = [random.randrange(4) for _ in holdout_labels]
fresh_accuracy = sum(p == l for p, l in zip(random_preds, holdout_labels)) / len(holdout_labels)

print(f"Transferred soft prompt accuracy: {transferred_accuracy:.3f}")
print(f"Random baseline accuracy:         {fresh_accuracy:.3f}")
print(f"Transfer gain: {transferred_accuracy - fresh_accuracy:+.3f}")
print()
print("Note: a soft prompt trained for only ~10 minutes will not be strong;")
print("the point is to measure how well the learned virtual tokens carry over")
print("to unseen complaint phrasings within the same 4-class routing task.")


In [ ]:
# Side-by-side parameter efficiency comparison across all three methods.

import numpy as np

methods = ["Full fine-tune", "LoRA (r=8)", "Soft prompts (10 virtual)"]

full_trainable = sum(p.numel() for p in
    AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=4
    ).parameters()
)

lora_trainable   = sum(p.numel() for p in peft_model.parameters()   if p.requires_grad)
prompt_trainable = sum(p.numel() for p in prefix_model.parameters() if p.requires_grad)

counts = [full_trainable, lora_trainable, prompt_trainable]
total  = full_trainable  # base model total params

print(f"{'Method':<30} {'Trainable':>12} {'% of total':>12}")
print("-" * 56)
for method, count in zip(methods, counts):
    pct = 100.0 * count / total
    print(f"{method:<30} {count:>12,} {pct:>11.3f}%")

print()
print("Key takeaway:")
print("  LoRA and soft prompts both freeze the base model.")
print("  LoRA modifies existing weight matrices (more expressive).")
print("  Soft prompts only add virtual tokens (simpler, fewer params).")
print("  For small models like DistilBERT, LoRA typically wins on accuracy.")

## Discussion (3 minutes)

1. QLoRA fits a 4-bit base model in 22MB vs 268MB for float32. What does that mean for
   which Barclays teams can now run fine-tuning? (Consider: a team that only has a
   laptop GPU vs a team with a dedicated GPU server.)

2. Soft prompts freeze the entire base model. If Barclays has a single DistilBERT base
   model and 20 different complaint categories (cards, mortgages, fraud, ...), how would
   you manage 20 separate sets of virtual token embeddings vs 20 separate LoRA adapter
   files?

3. When would you choose soft prompts over LoRA? (Hint: think about model size, data
   availability, and the cost of storing/serving multiple adapters.)

## Section 4: Capstone, PEFT Fine-Tuning on SageMaker GPU

The `scripts_topic6/` folder contains a training script that supports both LoRA and QLoRA
via a `--peft_method` argument. The script uses:

- a 4-class Barclays complaint-routing dataset defined inside train.py
  (labels: fraud and security, billing and charges, account access, general enquiry)
- HuggingFace Trainer with eval_strategy="epoch" and inline numpy metrics (no evaluate lib)
- PEFT LoraConfig with target_modules=["q_lin", "v_lin"] for DistilBERT
- BitsAndBytesConfig for QLoRA (4-bit NF4, fp16 compute)

The SageMaker job uses:
- HuggingFace estimator (GPU-only DLC, L1 from SAGEMAKER_LESSONS_LEARNED)
- transformers_version="4.56.2", pytorch_version="2.8.0", py_version="py312"
- instance_type="ml.g4dn.xlarge" (NVIDIA T4, 16GB VRAM, ~$0.74/hr)

In [ ]:
# Section 4 setup: verify the existing SageMaker session (sess, role, bucket, region
# were defined in Cell 1) and show the training script structure.
import os

print(f"Training will run on: ml.g4dn.xlarge (NVIDIA T4, 16GB VRAM)")
print(f"Region:               {region}")
print(f"Role:                 {role}")
print(f"Artifacts will land in: s3://{bucket}/topic6-peft/")
print()
print("scripts_topic6/ structure:")
for fname in sorted(os.listdir("scripts_topic6")):
    fpath = os.path.join("scripts_topic6", fname)
    size  = os.path.getsize(fpath)
    print(f"  {fname:<20} {size:>8} bytes")

In [ ]:
# Launch a LoRA fine-tuning job on SageMaker GPU.
# HuggingFace estimator is GPU-only (L1 from SAGEMAKER_LESSONS_LEARNED).

from sagemaker.huggingface import HuggingFace
import time

job_suffix    = int(time.time())
job_name_lora = f"topic6-lora-{job_suffix}"

estimator = HuggingFace(
    entry_point="train.py",
    source_dir="scripts_topic6",         # contains train.py and requirements.txt
    role=role,
    instance_type="ml.g4dn.xlarge",       # GPU only, HuggingFace DLC requirement
    instance_count=1,
    # Verified version matrix (CORE_TECHNOLOGIES_AND_DECISIONS.md).
    transformers_version="4.56.2",
    pytorch_version="2.8.0",
    py_version="py312",
    hyperparameters={
        "peft_method": "lora",
        "lora_r":      8,
        "lora_alpha":  16,
        "epochs":      3,
        "batch_size":  16,
        "lr":          2e-4,
    },
    output_path=f"s3://{bucket}/topic6-peft/output/",
    base_job_name=job_name_lora,
)

# wait=False, do not block the kernel while training runs.
estimator.fit(wait=False)
training_job_name = estimator.latest_training_job.name
print(f"Training job submitted: {training_job_name}")
print("Monitor in SageMaker console -> Training -> Training jobs")

In [ ]:
# Safety-net: run this if your kernel restarted after launching the training job.
# SKIP this cell if training_job_name is already defined.
if 'training_job_name' not in dir() or training_job_name is None:
    training_job_name = "<PASTE YOUR JOB NAME HERE>"
    print(f"Using safety-net training_job_name: {training_job_name}")

In [ ]:
# Poll training job status until complete.
import boto3
import time

sm_client = boto3.client("sagemaker", region_name=region)

print(f"Polling job: {training_job_name}")
print("(This cell blocks until the job is Complete or Failed.)")
print()

while True:
    resp   = sm_client.describe_training_job(TrainingJobName=training_job_name)
    status = resp["TrainingJobStatus"]
    print(f"  Status: {status}", end="\r", flush=True)
    if status in ("Completed", "Failed", "Stopped"):
        print(f"\nFinal status: {status}")
        break
    time.sleep(30)

if status == "Failed":
    reason = resp.get("FailureReason", "No reason provided")
    print(f"Failure reason: {reason}")

In [ ]:
# Read the metrics summary written by train.py and surface the final accuracy.
import boto3
import json

s3 = boto3.client("s3")

# Determine artifact prefix from the training job description.
resp      = sm_client.describe_training_job(TrainingJobName=training_job_name)
model_uri = resp["ModelArtifacts"]["S3ModelArtifacts"]
# model_uri: s3://bucket/.../output/model.tar.gz
# metrics.json is inside the tarball, retrieve from CloudWatch logs instead.

log_client  = boto3.client("logs", region_name=region)
log_group   = "/aws/sagemaker/TrainingJobs"

# Simpler: print the final status from the describe response. The Trainer logs
# eval_accuracy each epoch, visible in CloudWatch.
print(f"Training job: {training_job_name}")
print(f"Status      : {resp['TrainingJobStatus']}")
print()
print("Final accuracy is logged as 'eval_accuracy' in CloudWatch.")
print(f"CloudWatch log group : /aws/sagemaker/TrainingJobs")
print(f"Log stream prefix    : {training_job_name}/algo-1-")
print()
print("Tip: in the SageMaker console, open the training job and click 'View logs'")
print("to see epoch-by-epoch eval_accuracy printed by the Trainer.")

In [ ]:
# Handoff to Topic 7: save a pointer to the PEFT/LoRA fine-tuned model.
# Topic 7 loads this checkpoint and compresses it (quantization, pruning, distillation).

# --- S3 handoff helpers ---
import json, boto3

COURSE_PREFIX = "barclays-course"

def handoff_write(bucket, topic_n, artifact, obj):
    key = f"{COURSE_PREFIX}/topic_{topic_n}/{artifact}"
    boto3.client("s3").put_object(
        Bucket=bucket, Key=key,
        Body=json.dumps(obj, indent=2).encode("utf-8"),
    )
    print(f"Handoff written: s3://{bucket}/{key}")
    return key

# The PEFT GPU job writes model.tar.gz under s3://<bucket>/topic6-peft/output/.
try:
    peft_model_uri = resp["ModelArtifacts"]["S3ModelArtifacts"]
except (NameError, KeyError, TypeError):
    peft_model_uri = None

try:
    _job = training_job_name
except NameError:
    _job = None

model_pointer = {
    "model_tar_uri": peft_model_uri,
    "training_job_name": _job,
    "lora_config": {"lora_r": 8, "lora_alpha": 16},
    "label_map": label_map,
    "kind": "peft_lora",
}
handoff_write(bucket, 6, "model_pointer.json", model_pointer)
print()
if peft_model_uri is None:
    print("Note: no PEFT model URI recorded (GPU job not run in this kernel).")
    print("Topic 7 will fall back to a fresh DistilBERT if the URI is missing.")
print("Topic 7 will load this pointer and compress the model for production serving.")

## Day 2 Capstone: Barclays Complaint Intelligence System -- PEFT Assembly

This is a Tier 2 hard lab. The starter code provides YOUR CODE stubs that guide the
overall structure, but the specific PEFT method choice, hyperparameters, and SageMaker
estimator configuration are yours to design. There is no auto-verification cell: you
submit a SageMaker job and the instructor reviews the logs.

### Situation

The Barclays Machine Learning Platform team needs a complaint classifier that can be
deployed as a SageMaker endpoint. Your manager has given you one GPU hour and a choice
of PEFT method: LoRA, QLoRA, or prefix tuning. Pick the method you think is best for
this use case, justify your choice, and submit the training job.

### Task

Implement a function `train_peft_complaint_classifier` that:
- Accepts the PEFT method name, training data, and labels as arguments
- Fine-tunes a DistilBERT (or other encoder) classifier using that PEFT method
- Returns a summary dict with accuracy and parameter counts

Then submit the training job to SageMaker using the HuggingFace estimator.

### What to hand in

- Your completed function in this cell (or a separate notebook cell)
- A markdown cell explaining which PEFT method you chose and why
- The SageMaker training job name so the instructor can check the logs

### Constraints

- You may modify scripts_topic6/train.py or write your own training script
- Use ml.g4dn.xlarge, transformers_version="4.56.2", pytorch_version="2.8.0", py_version="py312"
- eval_strategy="epoch" (not evaluation_strategy)
- No evaluate library, use inline numpy metrics
- numpy<2 in requirements.txt
- No hardcoded credentials

In [ ]:
def train_peft_complaint_classifier(
    model_name: str,
    peft_method: str,
    train_texts: list,
    train_labels: list,
) -> dict:
    """
    Fine-tune a complaint classifier using the specified PEFT method.

    Returns a summary dict with accuracy, trainable_params, total_params.
    """
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        TrainingArguments, Trainer,
    )
    from peft import (
        LoraConfig, PromptTuningConfig, get_peft_model, TaskType,
    )
    from datasets import Dataset
    import numpy as np

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    enc = tokenizer(
        list(train_texts),
        truncation=True, padding="max_length", max_length=128,
    )
    enc["labels"] = list(train_labels)
    ds_full = Dataset.from_dict(enc)
    split = ds_full.train_test_split(test_size=0.2, seed=42)
    train_ds, eval_ds = split["train"], split["test"]
    train_ds.set_format("torch")
    eval_ds.set_format("torch")

    num_labels = len(set(train_labels))
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels,
    )

    if peft_method in ("lora", "qlora"):
        cfg = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=8, lora_alpha=16,
            target_modules=["q_lin", "v_lin"],
            lora_dropout=0.1, bias="none",
            modules_to_save=["classifier", "pre_classifier"],
        )
    elif peft_method == "prefix_tuning":
        cfg = PromptTuningConfig(
            task_type=TaskType.SEQ_CLS, num_virtual_tokens=20,
        )
    else:
        raise ValueError(f"Unknown peft_method: {peft_method}")
    peft_model = get_peft_model(base_model, cfg)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {"accuracy": float((preds == labels).mean())}

    training_args = TrainingArguments(
        output_dir="./capstone_out",
        num_train_epochs=1,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-4,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=20,
        report_to="none",
    )
    trainer = Trainer(
        model=peft_model, args=training_args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    eval_results = trainer.evaluate()
    accuracy = float(eval_results.get("eval_accuracy", 0.0))

    trainable_params = sum(
        p.numel() for p in peft_model.parameters() if p.requires_grad
    )
    total_params = sum(p.numel() for p in peft_model.parameters())
    return {
        "accuracy": accuracy,
        "trainable_params": trainable_params,
        "total_params": total_params,
    }

### Stretch (for fast finishers)

1. Submit TWO jobs: one with --peft_method=lora and one with --peft_method=qlora.
   Compare the training time and final accuracy. Is the accuracy gap worth the memory
   saving from QLoRA on a dataset this small?

2. Modify train.py to also log the number of trainable parameters as a CloudWatch
   metric (use SageMaker's logging_steps and a custom MetricsDefinition on the
   estimator). Plot trainable params vs eval_accuracy for r = 4, 8, 16.

### Homework Extension

LoRA adapters can be merged back into the base model weights after training using
`model.merge_and_unload()`. Research this PEFT method and explain:
  (a) When would you merge adapters vs keep them separate?
  (b) What happens to inference latency after merging?
  (c) Write a short code snippet (can be pseudocode) showing the merge step and
      how you would save the merged model to S3 for SageMaker endpoint deployment.

In [ ]:
# Homework Extension starter: pseudocode for merging LoRA adapters back into the base.
# (This cell is illustrative, the merge call requires the trained peft_model.)
#
# from peft import PeftModel
# trained = PeftModel.from_pretrained(base_model, "/opt/ml/model")
# merged  = trained.merge_and_unload()       # produces a regular HF model
# merged.save_pretrained("/opt/ml/model_merged")
# # Upload merged/ folder to s3://bucket/topic6-peft/merged/ and deploy as an endpoint.
print("See comments above for the merge_and_unload pseudocode.")

## Topic 6 Wrap-Up

### What you built in this topic

- Applied the HuggingFace PEFT library (LoraConfig plus get_peft_model) to DistilBERT
  for complaint classification, with only ~0.5-1% of parameters trainable.
- Explored QLoRA: 4-bit NF4 quantized base model plus LoRA adapters, fitting larger
  models on the same T4 GPU.
- Configured soft prompts / prefix tuning as an even lighter alternative to LoRA.
- Launched and monitored a GPU fine-tuning job on SageMaker (ml.g4dn.xlarge).
- Compared trainable parameter counts across LoRA, QLoRA, soft prompts, and full FT.

### Key PEFT takeaways

- LoraConfig plus get_peft_model works on any HuggingFace model in 5 lines.
- target_modules=["q_lin", "v_lin"] for DistilBERT attention. Check model.named_modules()
  for other architectures.
- QLoRA requires a CUDA GPU, use the HuggingFace estimator on ml.g4dn.xlarge.
- Soft prompts have the fewest trainable parameters but underfit small models.
- eval_strategy="epoch" (not evaluation_strategy), transformers 4.41+ required.
- Never use the evaluate library, use inline numpy compute_metrics.

### Bridge to Topic 7

Next: Topic 7 - Quantization. We take a pretrained classifier and shrink its
footprint with post-training quantization, pruning, and distillation so it can
run on smaller, cheaper instances at inference time.

### Homework

Complete the Homework Extensions from Lab 1, Lab 2, and the Capstone before moving on.

## Day 2 Open-Ended Capstone (Tier 3 - ~45 min, individual or pair)

### Situation
You have completed the core training arc of the course: pre-trained models from the
Hub (Topic 3), full fine-tuning (Topic 4), transfer learning (Topic 5), and the PEFT
library (Topic 6, this notebook). The Barclays ML Platform team now asks you to design
and document the production PEFT strategy for a new complaint routing system.

### Task
Design -- do not just implement -- the end-to-end PEFT pipeline for Barclays.
Your deliverable is a markdown cell (below) plus one working proof-of-concept code cell.

Answer these questions in the markdown cell:
1. Which base model would you choose and why? (DistilBERT or another open model)
2. Which PEFT method would you use (LoRA, QLoRA, soft prompts) and what rank/config?
3. How many trainable parameters does your choice require?
4. What is your estimated GPU cost for one training run on ml.g4dn.xlarge at $0.74/hr?
5. How would you version and serve multiple adapter variants without redeploying the base?

### Action
No scaffolding. No YOUR CODE stubs. Design first, then implement your proof-of-concept.

### Stretch
Submit your proof-of-concept as a SageMaker job using scripts_topic6/train.py.
Compare two PEFT methods and write a one-paragraph recommendation for the Barclays team.

In [ ]:
# Day 2 Tier 3 Capstone: Your PEFT design and proof-of-concept
# No starter code. Write your design notes as comments, then implement.

# Design notes:
# Base model:
# PEFT method:
# Rank / config:
# Trainable params:
# Estimated cost:
# Versioning strategy:

# Proof-of-concept implementation:


## The Training Arc, Complete

You have built the Barclays Customer Support Intelligence System training pipeline:

| Topic | What you built | Key insight |
|-------|---------------|-------------|
| Topic 3 | HuggingFace Hub inference pipeline | Pre-trained beats from-scratch |
| Topic 4 | Full fine-tuning with catastrophic forgetting | Adapting costs memory |
| Topic 5 | Transfer learning (frozen encoder + trained head) | Freeze expensive, train cheap |
| Topic 6 | PEFT library LoRA + QLoRA on DistilBERT | Library beats hand-rolled |

**Next preview**: Topic 7 - Quantization, pruning, and distillation -- making the
trained model smaller and faster for production serving without sacrificing accuracy.

### What Comes Next

Your PEFT-adapted model is accurate -- but it is still float32 and larger than a production endpoint wants.
Topic 7 teaches you to compress it without losing accuracy:
Post-Training Quantization, Quantization-Aware Training, structured pruning, and knowledge distillation.
Then you will serve it from a real SageMaker endpoint -- replacing the GPT-4o API call from Topic 1.
